In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [3]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from datasets import load_dataset

#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import json
import re
import numpy as np
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Set, Optional
from dataclasses import dataclass, field
from tqdm import tqdm

import torch
import networkx as nx
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("All imports successful.")

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


All imports successful.


In [5]:
# Load a subset of SQuAD v2 validation set
NUM_SAMPLES = 100  # Adjust as needed
dataset = load_dataset("rajpurkar/squad_v2", split=f"validation[:{NUM_SAMPLES}]")

# Extract unique passages (SQuAD has many questions per passage)
# We keep track of passage_id for each question
passage_map = {}  # context_text -> passage_id
passages = []     # list of unique passage texts
questions = []
ground_truths = []
question_passage_ids = []  # which passage each question belongs to

for row in dataset:
    ctx = row["context"]
    if ctx not in passage_map:
        passage_map[ctx] = len(passages)
        passages.append(ctx)
    
    questions.append(row["question"])
    question_passage_ids.append(passage_map[ctx])
    
    if len(row["answers"]["text"]) > 0:
        ground_truths.append(row["answers"]["text"][0])
    else:
        ground_truths.append("")  # unanswerable

print(f"Total questions: {len(questions)}")
print(f"Unique passages: {len(passages)}")
print(f"Unanswerable questions: {sum(1 for gt in ground_truths if gt == '')}")

Total questions: 100
Unique passages: 17
Unanswerable questions: 55


In [6]:
llm = HuggingFacePipeline.from_model_id(
    # model_id="/mnt/nas/shuvranshu/huggingface_cache/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", 
    model_id="meta-llama/Llama-3.1-8B",
    # model_id="/mnt/nas/shuvranshu/finetune/merged_model",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "max_length": None,
        "do_sample": False,
    },
    device_map={"":0}
)


Loading weights: 100%|██████████| 291/291 [00:01<00:00, 252.77it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [7]:

def call_llm(prompt: str) -> str:
    response = llm.invoke(prompt)
    # Base model returns the full prompt + completion,
    # so extract only the generated part
    if prompt in response:
        response = response[len(prompt):]
    return response.strip()

In [8]:
def parse_json_from_llm(response: str) -> dict:
    """Robustly parse JSON from LLM output."""
    # Try to find JSON in the response
    # First try direct parse
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass
    
    # Try to find JSON block in markdown code fences
    json_match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Try to find anything that looks like a JSON object
    json_match = re.search(r'(\{.*\})', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass
    
    # Fallback
    return {}

In [9]:
# Quick test
test_response = call_llm("Say 'hello' in JSON format like {\"greeting\": \"hello\"}")
print("LLM test:", test_response)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM test: and get a response like {"greeting": "hello", "name": "John Doe", "age": 30}. The response will be in JSON format.
```
curl -X POST -H "Content-Type: application/json" -d '{"greeting": "hello"}' http://localhost:8080/greeting
```
```
{
  "greeting": "hello",
  "name": "John Doe",
  "age": 30
}
```
## How to run

  1. Clone the repository: ``` git clone https://github.com/robertoacosta/greeting-service.git ```
  2. Build the project: ```./mvnw clean package ```
  3. Run the application: ``` java -jar target/greeting-service-0.0.1-SNAPSHOT.jar ```

## How to test

  1. Run the application: ``` java -jar target/greeting-service-0.0.1-SNAPSHOT.jar ```
  2. Run the tests: ```./mvnw test ```

## How to deploy

  1. Build the project: ```./mvnw clean package ```
  2. Run the application: ``` java -jar target/greeting-service


In [10]:
retrieval_encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda:0"
)
print(f"Retrieval encoder loaded. Embedding dim: {retrieval_encoder.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 290.48it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Retrieval encoder loaded. Embedding dim: 384


In [11]:
@dataclass
class HippoRAGIndex:
    """The hippocampal index: a knowledge graph with passage mappings."""
    
    graph: nx.Graph = field(default_factory=nx.Graph)
    
    # Node -> set of passage IDs (the P matrix from the paper)
    node_to_passages: Dict[str, Set[int]] = field(default_factory=lambda: defaultdict(set))
    
    # All passages
    passages: List[str] = field(default_factory=list)
    
    # Node embeddings cache
    node_embeddings: Optional[np.ndarray] = None
    node_list: List[str] = field(default_factory=list)  # ordered list matching embeddings
    
    def add_triple(self, subj: str, rel: str, obj: str, passage_id: int):
        """Add a triple to the KG and update passage mappings."""
        subj_norm = subj.lower().strip()
        obj_norm = obj.lower().strip()
        
        if not subj_norm or not obj_norm:
            return
        
        # Add nodes and edge
        self.graph.add_node(subj_norm)
        self.graph.add_node(obj_norm)
        self.graph.add_edge(subj_norm, obj_norm, relation=rel)
        
        # Update passage mappings
        self.node_to_passages[subj_norm].add(passage_id)
        self.node_to_passages[obj_norm].add(passage_id)
    
    def add_synonymy_edges(self, encoder: SentenceTransformer, threshold: float = 0.8):
        """Add synonymy edges between nodes with high embedding similarity.
        This mimics the parahippocampal regions detecting synonyms."""
        
        self.node_list = list(self.graph.nodes())
        if len(self.node_list) == 0:
            return
        
        print(f"Computing embeddings for {len(self.node_list)} nodes...")
        self.node_embeddings = encoder.encode(
            self.node_list, 
            show_progress_bar=True,
            normalize_embeddings=True,
            batch_size=256
        )
        
        # Compute pairwise cosine similarity in batches to save memory
        synonymy_count = 0
        batch_size = 1000
        n = len(self.node_list)
        
        print("Finding synonymy edges...")
        for i in tqdm(range(0, n, batch_size)):
            end_i = min(i + batch_size, n)
            batch_emb = self.node_embeddings[i:end_i]
            
            # Compare with all nodes after current batch start
            for j in range(i, n, batch_size):
                end_j = min(j + batch_size, n)
                other_emb = self.node_embeddings[j:end_j]
                
                sims = np.dot(batch_emb, other_emb.T)
                
                for ii in range(sims.shape[0]):
                    for jj in range(sims.shape[1]):
                        abs_i = i + ii
                        abs_j = j + jj
                        if abs_i >= abs_j:
                            continue  # skip self and duplicates
                        if sims[ii, jj] > threshold:
                            node_a = self.node_list[abs_i]
                            node_b = self.node_list[abs_j]
                            if not self.graph.has_edge(node_a, node_b):
                                self.graph.add_edge(node_a, node_b, relation="synonymy")
                                synonymy_count += 1
        
        print(f"Added {synonymy_count} synonymy edges.")
    
    def get_node_specificity(self, node: str) -> float:
        """Node specificity = 1 / |passages containing this node|.
        Neurobiologically plausible IDF alternative (Section 2.3)."""
        n_passages = len(self.node_to_passages.get(node, set()))
        if n_passages == 0:
            return 0.0
        return 1.0 / n_passages
    
    def stats(self):
        """Print KG statistics."""
        triple_edges = sum(1 for _, _, d in self.graph.edges(data=True) if d.get('relation') != 'synonymy')
        syn_edges = sum(1 for _, _, d in self.graph.edges(data=True) if d.get('relation') == 'synonymy')
        print(f"Nodes: {self.graph.number_of_nodes()}")
        print(f"Triple edges: {triple_edges}")
        print(f"Synonymy edges: {syn_edges}")
        print(f"Total edges: {self.graph.number_of_edges()}")
        print(f"Passages: {len(self.passages)}")

print("KG data structures defined.")

KG data structures defined.


In [ ]:
# def passage_ner(passage: str) -> List[str]:
#     """Extract named entities from a passage using the LLM.
#     Follows the prompt in Figure 7 of the paper."""
    
#     prompt = f"""Your task is to extract named entities from the given paragraph.
# Respond with a JSON list of entities.

# Paragraph:
# ```
# Radio City
# Radio City is India's first private FM radio station and was started on 3 July 2001. It plays Hindi, English and regional songs. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.
# ```
# {{"named_entities": ["Radio City", "India", "3 July 2001", "Hindi", "English", "May 2008", "PlanetRadiocity.com"]}}

# Paragraph:
# ```
# {passage}
# ```"""
    
#     response = call_llm(prompt)
#     parsed = parse_json_from_llm(response)
#     entities = parsed.get("named_entities", [])
    
#     # Ensure we return a list of strings
#     if isinstance(entities, list):
#         return [str(e) for e in entities if e]
#     return []


# # Test
# test_entities = passage_ner(passages[0][:500])
# print(f"Test NER result: {test_entities}")

Test NER result: []


In [13]:
import spacy
nlp = spacy.load("en_core_web_sm")

def passage_ner(passage: str) -> List[str]:
    """Extract named entities using spaCy."""
    doc = nlp(passage)
    entities = [ent.text for ent in doc.ents]
    # Also add noun chunks as fallback
    if len(entities) < 3:
        for chunk in doc.noun_chunks:
            if chunk.text not in entities:
                entities.append(chunk.text)
    return entities


def passage_openie(passage: str, entities: List[str]) -> List[Tuple[str, str, str]]:
    """Extract triples using spaCy dependency parsing."""
    doc = nlp(passage)
    triples = []
    
    for sent in doc.sents:
        for token in sent:
            if token.pos_ == "VERB":
                # Find subjects
                subjects = [w for w in token.children if w.dep_ in ("nsubj", "nsubjpass")]
                # Find objects
                objects = [w for w in token.children if w.dep_ in ("dobj", "pobj", "attr")]
                
                # Also check prepositional objects
                for child in token.children:
                    if child.dep_ == "prep":
                        for grandchild in child.children:
                            if grandchild.dep_ == "pobj":
                                objects.append(grandchild)
                
                for subj in subjects:
                    # Expand to full noun phrase
                    subj_span = _get_span(subj)
                    for obj in objects:
                        obj_span = _get_span(obj)
                        triples.append((subj_span, token.lemma_, obj_span))
    
    # Also create entity co-occurrence triples
    doc_ents = [ent for ent in doc.ents]
    for i in range(len(doc_ents)):
        for j in range(i + 1, len(doc_ents)):
            if doc_ents[i].sent == doc_ents[j].sent:
                triples.append((doc_ents[i].text, "related_to", doc_ents[j].text))
    
    return triples


def _get_span(token):
    """Get the full noun phrase span for a token."""
    subtree = list(token.subtree)
    start = subtree[0].i
    end = subtree[-1].i + 1
    return token.doc[start:end].text


def query_ner(question: str) -> List[str]:
    """Extract named entities from a query using spaCy."""
    doc = nlp(question)
    entities = [ent.text for ent in doc.ents]
    
    # Fallback to noun chunks if no named entities found
    if not entities:
        entities = [chunk.text for chunk in doc.noun_chunks 
                    if chunk.root.pos_ in ("NOUN", "PROPN")]
    
    # Last resort: use important words
    if not entities:
        entities = [token.text for token in doc 
                    if token.pos_ in ("NOUN", "PROPN") and not token.is_stop]
    
    return entities

In [ ]:
# def passage_openie(passage: str, entities: List[str]) -> List[Tuple[str, str, str]]:
#     """Extract RDF triples from a passage given named entities.
#     Follows the prompt in Figure 9 of the paper."""
    
#     ner_json = json.dumps(entities)
    
#     prompt = f"""Your task is to construct an RDF (Resource Description Framework) graph from the given passages and named entity lists.
# Respond with a JSON list of triples, with each triple representing a relationship in the RDF graph.
# Pay attention to the following requirements:
# - Each triple should contain at least one, but preferably two, of the named entities in the list for each passage.
# - Clearly resolve pronouns to their specific names to maintain clarity.

# Convert the paragraph into a JSON dict, it has a named entity list and a triple list.

# Paragraph:
# ```
# Radio City
# Radio City is India's first private FM radio station and was started on 3 July 2001. It plays Hindi, English and regional songs. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.
# ```
# {{"named_entities": ["Radio City", "India", "3 July 2001", "Hindi", "English", "May 2008", "PlanetRadiocity.com"]}}
# {{"triples":
#  [
#  ["Radio City", "located in", "India"],
#  ["Radio City", "is", "private FM radio station"],
#  ["Radio City", "started on", "3 July 2001"],
#  ["Radio City", "plays songs in", "Hindi"],
#  ["Radio City", "plays songs in", "English"],
#  ["Radio City", "forayed into", "New Media"],
#  ["Radio City", "launched", "PlanetRadiocity.com"],
#  ["PlanetRadiocity.com", "launched in", "May 2008"],
#  ["PlanetRadiocity.com", "is", "music portal"],
#  ["PlanetRadiocity.com", "offers", "news"],
#  ["PlanetRadiocity.com", "offers", "videos"],
#  ["PlanetRadiocity.com", "offers", "songs"]
#  ]
# }}

# Convert the paragraph into a JSON dict, it has a named entity list and a triple list.
# Paragraph:
# ```
# {passage}
# ```
# {{"named_entities": {ner_json}}}"""
    
#     response = call_llm(prompt)
#     parsed = parse_json_from_llm(response)
#     triples_raw = parsed.get("triples", [])
    
#     triples = []
#     if isinstance(triples_raw, list):
#         for t in triples_raw:
#             if isinstance(t, (list, tuple)) and len(t) == 3:
#                 triples.append((str(t[0]), str(t[1]), str(t[2])))
    
#     return triples


# # Test
# test_triples = passage_openie(passages[0][:500], test_entities)
# print(f"Test OpenIE result ({len(test_triples)} triples):")
# for t in test_triples[:5]:
#     print(f"  {t}")

In [ ]:
# def query_ner(question: str) -> List[str]:
#     """Extract named entities from a query.
#     Follows the prompt in Figure 8 of the paper."""
    
#     prompt = f"""You're a very effective entity extraction system. Please extract all named entities that are important for solving the questions below. Place the named entities in JSON format.

# Question: Which magazine was started first Arthur's Magazine or First for Women?
# {{"named_entities": ["First for Women", "Arthur's Magazine"]}}

# Question: {question}"""
    
#     response = call_llm(prompt)
#     parsed = parse_json_from_llm(response)
#     entities = parsed.get("named_entities", [])
    
#     if isinstance(entities, list):
#         return [str(e) for e in entities if e]
#     return []


# # Test
# test_q_entities = query_ner(questions[0])
# print(f"Question: {questions[0]}")
# print(f"Query NER: {test_q_entities}")

In [14]:
# Initialize the hippocampal index
hippo_index = HippoRAGIndex()
hippo_index.passages = passages

# Step 1: Process each passage through NER + OpenIE
print("=== Step 1: NER + OpenIE for each passage ===")
all_passage_entities = []  # Store for debugging
all_passage_triples = []

for pid, passage in enumerate(tqdm(passages, desc="Indexing passages")):
    try:
        # NER
        entities = passage_ner(passage)
        all_passage_entities.append(entities)
        
        # OpenIE
        triples = passage_openie(passage, entities)
        all_passage_triples.append(triples)
        
        # Add triples to KG
        for subj, rel, obj in triples:
            hippo_index.add_triple(subj, rel, obj, pid)
        
        # Also add bare entities to passage mapping (even if not in triples)
        for ent in entities:
            ent_norm = ent.lower().strip()
            if ent_norm:
                hippo_index.graph.add_node(ent_norm)
                hippo_index.node_to_passages[ent_norm].add(pid)
    
    except Exception as e:
        print(f"Error on passage {pid}: {e}")
        all_passage_entities.append([])
        all_passage_triples.append([])

print("\nKG after OpenIE:")
hippo_index.stats()

=== Step 1: NER + OpenIE for each passage ===


Indexing passages: 100%|██████████| 17/17 [00:01<00:00, 14.29it/s]


KG after OpenIE:
Nodes: 394
Triple edges: 965
Synonymy edges: 0
Total edges: 965
Passages: 17


In [15]:
# Step 2: Add synonymy edges (parahippocampal regions)
# The paper uses threshold tau = 0.8
print("=== Step 2: Adding synonymy edges ===")
hippo_index.add_synonymy_edges(retrieval_encoder, threshold=0.8)

print("\nFinal KG statistics:")
hippo_index.stats()

=== Step 2: Adding synonymy edges ===
Computing embeddings for 394 nodes...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.84it/s]


Finding synonymy edges...


100%|██████████| 1/1 [00:00<00:00,  4.39it/s]

Added 79 synonymy edges.

Final KG statistics:
Nodes: 394
Triple edges: 965
Synonymy edges: 79
Total edges: 1044
Passages: 17


In [16]:
# Inspect some triples
print("Sample triples from the KG:")
for i, (u, v, d) in enumerate(hippo_index.graph.edges(data=True)):
    if d.get('relation') != 'synonymy':
        print(f"  ({u}) --[{d.get('relation', '?')}]--> ({v})")
    if i > 15:
        break

print(f"\nSample synonymy edges:")
syn_count = 0
for u, v, d in hippo_index.graph.edges(data=True):
    if d.get('relation') == 'synonymy':
        print(f"  {u} <-> {v}")
        syn_count += 1
    if syn_count > 5:
        break

Sample triples from the KG:
  (they) --[descend]--> (norse)
  (they) --[form]--> (which)
  (they) --[adopt]--> (the gallo-romance language of the frankish land they settled)
  (they) --[enter]--> (the byzantine empire and then armenia)
  (they) --[fight]--> (byzantine service)
  (they) --[fight]--> (sicily)
  (they) --[base]--> (malatya and edessa)
  (they) --[base]--> (the byzantine duke of antioch, isaac komnenos)
  (they) --[lend]--> (their ethnicity to the name of their castle: afranji, meaning "franks)
  (they) --[join]--> (the fleet that had previously conquered corfu and attacked dyrrachium from land and sea, devastating everything along the way)
  (they) --[take]--> (ioannina and some minor cities in southwestern macedonia and thessaly)
  (they) --[lose]--> (dyrrachium, valona, and butrint)
  (they) --[lose]--> (1085)
  (they) --[lose]--> (the death of robert)
  (they) --[occupy]--> (petrela, the citadel of mili at the banks of the river deabolis, gllavenica (ballsh), kanina an

In [17]:
#retrieval
def hipporag_retrieve(
    question: str,
    hippo_index: HippoRAGIndex,
    encoder: SentenceTransformer,
    top_k: int = 5,
    damping: float = 0.5  # Paper uses 0.5 (probability of restarting from query nodes)
) -> List[Tuple[int, float]]:
    """
    HippoRAG online retrieval.
    
    Returns: list of (passage_id, score) tuples, sorted by score descending.
    """
    
    # --- Step 1: Query NER (Neocortex) ---
    q_entities = query_ner(question)
    
    if not q_entities:
        # Fallback: use the question words as entities
        q_entities = [question]
    
    # --- Step 2: Link query entities to KG nodes (Parahippocampal Regions) ---
    if hippo_index.node_embeddings is None or len(hippo_index.node_list) == 0:
        return []
    
    q_embeddings = encoder.encode(
        q_entities, 
        normalize_embeddings=True
    )
    
    query_nodes = []
    for q_emb in q_embeddings:
        # Find the KG node with highest cosine similarity
        sims = np.dot(hippo_index.node_embeddings, q_emb)
        best_idx = np.argmax(sims)
        best_node = hippo_index.node_list[best_idx]
        best_sim = sims[best_idx]
        
        # Only link if similarity is reasonable
        if best_sim > 0.3:
            query_nodes.append(best_node)
    
    if not query_nodes:
        return []
    
    # Deduplicate query nodes
    query_nodes = list(set(query_nodes))
    
    # --- Step 3: Node specificity weighting ---
    # Multiply each query node's initial probability by its specificity
    personalization = {}
    for qn in query_nodes:
        specificity = hippo_index.get_node_specificity(qn)
        personalization[qn] = specificity
    
    # Normalize personalization to sum to 1
    total = sum(personalization.values())
    if total > 0:
        personalization = {k: v / total for k, v in personalization.items()}
    else:
        # Equal probability if all specificity is 0
        personalization = {qn: 1.0 / len(query_nodes) for qn in query_nodes}
    
    # --- Step 4: Personalized PageRank (Hippocampus) ---
    # The paper uses damping factor 0.5 (alpha in networkx = 1 - damping)
    # In networkx: alpha is the probability of following an edge (not restarting)
    # Paper's damping = probability of restarting = 0.5
    # So networkx alpha = 1 - 0.5 = 0.5
    try:
        ppr_scores = nx.pagerank(
            hippo_index.graph,
            alpha=1.0 - damping,  # alpha = probability of following edges
            personalization=personalization,
            max_iter=100,
            tol=1e-6
        )
    except nx.PowerIterationFailedConvergence:
        # Fallback: just use query node neighborhoods
        ppr_scores = personalization
    
    # --- Step 5: Aggregate node scores to passage scores ---
    # p_score[passage_id] = sum of PPR scores of nodes in that passage
    passage_scores = np.zeros(len(hippo_index.passages))
    
    for node, score in ppr_scores.items():
        if node in hippo_index.node_to_passages:
            for pid in hippo_index.node_to_passages[node]:
                passage_scores[pid] += score
    
    # --- Step 6: Rank and return top-k ---
    ranked_pids = np.argsort(passage_scores)[::-1][:top_k]
    results = [(int(pid), passage_scores[pid]) for pid in ranked_pids if passage_scores[pid] > 0]
    
    return results


print("Retrieval function defined.")

Retrieval function defined.


In [18]:
# Test retrieval on the first question
print(f"Question: {questions[0]}")
print(f"Ground truth: {ground_truths[0]}")
print(f"Gold passage ID: {question_passage_ids[0]}")
print()

results = hipporag_retrieve(questions[0], hippo_index, retrieval_encoder, top_k=5)
print("Retrieved passages (by rank):")
for rank, (pid, score) in enumerate(results):
    is_gold = "[GOLD]" if pid == question_passage_ids[0] else ""
    print(f"  Rank {rank+1}: passage {pid} (score={score:.6f}) {is_gold}")
    print(f"    Preview: {hippo_index.passages[pid][:120]}...")

Question: In what country is Normandy located?
Ground truth: France
Gold passage ID: 0

Retrieved passages (by rank):
  Rank 1: passage 1 (score=0.677826) 
    Preview: The Norman dynasty had a major political, cultural and military impact on medieval Europe and even the Near East. The No...
  Rank 2: passage 0 (score=0.674929) [GOLD]
    Preview: The Normans (Norman: Nourmands; French: Normands; Latin: Normanni) were the people who in the 10th and 11th centuries ga...
  Rank 3: passage 14 (score=0.666779) 
    Preview: The Normans were in contact with England from an early date. Not only were their original Viking brethren still ravaging...
  Rank 4: passage 6 (score=0.663466) 
    Preview: The Normans thereafter adopted the growing feudal doctrines of the rest of France, and worked them into a functional hie...
  Rank 5: passage 4 (score=0.650322) 
    Preview: Before Rollo's arrival, its populations did not differ from Picardy or the Île-de-France, which were considered "Frankis...


In [19]:
def answer_question(question: str, context_passages: List[str]) -> str:
    """Use the LLM to answer a question given retrieved passages."""
    
    # Combine passages into context
    context = "\n\n".join(context_passages)
    
    prompt = f"""You are a factual QA assistant. Answer the question using ONLY the provided context.

Instructions:
1. Use ONLY information from the context below.
2. Do NOT use external knowledge.
3. If the answer cannot be found in the context, respond with exactly: "unanswerable"
4. Keep your answer concise — ideally a short phrase or a few words.
5. Do not repeat the question or add explanations.

Context:
{context}

Question: {question}

Answer:"""
    
    response = call_llm(prompt)
    
    # Clean up the response
    answer = response.strip()
    # Take first line only
    answer = answer.split("\n")[0].strip()
    # Remove common prefixes
    for prefix in ["Answer:", "The answer is", "A:"]:
        if answer.lower().startswith(prefix.lower()):
            answer = answer[len(prefix):].strip()
    
    return answer


# Test
if results:
    test_passages = [hippo_index.passages[pid] for pid, _ in results[:3]]
    test_answer = answer_question(questions[0], test_passages)
    print(f"Question: {questions[0]}")
    print(f"Predicted: {test_answer}")
    print(f"Ground truth: {ground_truths[0]}")

Question: In what country is Normandy located?
Predicted: France
Ground truth: France


In [23]:
# Run HippoRAG retrieval + QA on all questions
TOP_K = 5

all_predictions = []
all_retrieved_pids = []
all_retrieved_contexts = []

# Retrieval metrics
recall_at_2 = 0
recall_at_5 = 0

for idx in tqdm(range(len(questions)), desc="Evaluating"):
    question = questions[idx]
    gold_pid = question_passage_ids[idx]
    
    # --- Retrieval ---
    retrieved = hipporag_retrieve(question, hippo_index, retrieval_encoder, top_k=TOP_K)
    retrieved_pids = [pid for pid, _ in retrieved]
    all_retrieved_pids.append(retrieved_pids)
    
    # Retrieval recall
    if gold_pid in retrieved_pids[:2]:
        recall_at_2 += 1
    if gold_pid in retrieved_pids[:5]:
        recall_at_5 += 1
    
    # --- QA ---
    if retrieved_pids:
        context_passages = [hippo_index.passages[pid] for pid in retrieved_pids[:3]]
    else:
        context_passages = ["No relevant context found."]
    
    all_retrieved_contexts.append(context_passages)
    
    answer = answer_question(question, context_passages)
    print(f"Q{idx}: {answer}")
    all_predictions.append(answer)
    
    if idx < 10:  # Print first 10 for debugging
        print(f"\nQ{idx}: {question}")
        print(f"  Predicted: {answer}")
        print(f"  Ground truth: {ground_truths[idx]}")
        print(f"  Gold passage retrieved: {gold_pid in retrieved_pids[:5]}")

print(f"\n{'='*60}")
print(f"Retrieval Results:")
print(f"  Recall@2: {recall_at_2}/{len(questions)} = {recall_at_2/len(questions):.4f}")
print(f"  Recall@5: {recall_at_5}/{len(questions)} = {recall_at_5/len(questions):.4f}")

Evaluating:   1%|          | 1/100 [00:00<00:18,  5.23it/s]

Q0: France

Q0: In what country is Normandy located?
  Predicted: France
  Ground truth: France
  Gold passage retrieved: True


Evaluating:   2%|▏         | 2/100 [00:00<00:21,  4.61it/s]

Q1: unanswerable

Q1: When were the Normans in Normandy?
  Predicted: unanswerable
  Ground truth: 10th and 11th centuries
  Gold passage retrieved: True


Evaluating:   4%|▍         | 4/100 [00:00<00:23,  4.03it/s]

...

Q97: 1066


Evaluating:  99%|█████████▉| 99/100 [00:43<00:00,  2.96it/s]

Q98: The Byzantines


Evaluating: 100%|██████████| 100/100 [00:43<00:00,  2.27it/s]

Q99: unanswerable

Retrieval Results:
  Recall@2: 63/100 = 0.6300
  Recall@5: 79/100 = 0.7900


In [24]:
#EM
# all_predictions = []
# all_retrieved_pids = []
# all_retrieved_contexts = []
def normalize(text):
    text = text.strip().lower()
    if text == "unanswerable":
        return ""
    return text

correct = sum([
    normalize(pred) == normalize(gt)
    for pred, gt in zip(all_predictions, ground_truths)
])

total = len(ground_truths)

print("Correct:", correct)
print("Accuracy:", correct / total)

Correct: 31
Accuracy: 0.31


In [25]:
#substring match
def normalize(text):
    text = text.strip().lower()
    if text in ["unanswerable", "no answer", "no_answer", "n/a"]:
        return ""
    return text

correct = sum([
    normalize(gt) in normalize(pred)
    for pred, gt in zip(all_predictions, ground_truths)
])

print("Accuracy:", correct / len(ground_truths))

Accuracy: 0.77


In [26]:
from collections import Counter

def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()

    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_common = sum(common.values())

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)

In [27]:
f1_scores = []

for pred, gt in zip(all_predictions, ground_truths):
    

    if pred is None or pred.strip() == "":
        continue   # skip bad outputs

    # em_scores.append(compute_em(pred, gt))
    f1_scores.append(compute_f1(pred, gt))

# avg_em = sum(em_scores) / len(em_scores)
avg_f1 = sum(f1_scores) / len(f1_scores)

# print("Exact Match (EM):", avg_em)
print("F1 Score:", avg_f1)

F1 Score: 0.2041206433363296


In [28]:

dataset_ragas = []

for q, ctx, ans, ref in zip(questions, all_retrieved_contexts, all_predictions, ground_truths):
    dataset_ragas.append({
        "user_input": q,
        "retrieved_contexts": ctx if isinstance(ctx, list) else [ctx],
        "response": ans,
        "reference": ref
    })

In [29]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset_ragas)

In [30]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from langchain_community.embeddings import OllamaEmbeddings
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness


In [31]:
json_prompt = """
You are a strict JSON output generator used for evaluation in RAG systems.
You must always reply ONLY with a valid JSON object — no explanations, no extra text.

If the question asks for a score, output: {"score": <float between 0 and 1>}
If the question asks for a label (e.g., yes/no), output: {"label": "yes"} or {"label": "no"}
Do not include extra keys, comments, or markdown fences.
"""
langchain_llm= ChatOllama(model="mistral",temperature=0,system=json_prompt)


langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")

/tmp/ipykernel_2146746/241535895.py:9: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  langchain_llm= ChatOllama(model="mistral",temperature=0,system=json_prompt)
/tmp/ipykernel_2146746/241535895.py:12: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")


In [32]:
result = evaluate(dataset=evaluation_dataset,
                  batch_size=2,
                #   metrics=[ LLMContextRecall(), Faithfulness(),FactualCorrectness()],
                  metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
                  llm=langchain_llm,embeddings=langchain_embeddings)
print(result)

Evaluating:  40%|████      | 120/300 [06:33<08:50,  2.95s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt n_l_i_statement_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[121]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating: 100%|██████████| 300/300 [18:26<00:00,  3.69s/it]


{'context_recall': 0.3949, 'faithfulness': 0.5875, 'factual_correctness(mode=f1)': 0.4528}


In [33]:
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

qa_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cuda:0")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3911.71it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [36]:
import string

def normalize_answer(text: str) -> str:
    """Normalize answer text for comparison (SQuAD-style)."""
    if not text:
        return ""
    
    text = text.lower().strip()
    
    # Map model's "unanswerable" variants to empty string
    unanswerable_phrases = [
        "unanswerable", "no answer", "no_answer", "n/a",
        "cannot be determined", "not mentioned", "not found",
        "cannot be answered", "the answer cannot be found"
    ]
    for phrase in unanswerable_phrases:
        if phrase in text:
            return ""
    
    # Remove articles
    text = re.sub(r'\b(a|an|the)\b', ' ', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Normalize whitespace
    text = ' '.join(text.split())
    
    return text


def compute_exact_match(pred: str, gt: str) -> float:
    """Exact match score."""
    return 1.0 if normalize_answer(pred) == normalize_answer(gt) else 0.0


def compute_f1(pred: str, gt: str) -> float:
    """Token-level F1 score (SQuAD-style)."""
    pred_tokens = normalize_answer(pred).split()
    gt_tokens = normalize_answer(gt).split()
    
    # Handle both empty (both unanswerable) -> perfect match
    if not pred_tokens and not gt_tokens:
        return 1.0
    
    # One empty, other not -> 0
    if not pred_tokens or not gt_tokens:
        return 0.0
    
    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_common = sum(common.values())
    
    if num_common == 0:
        return 0.0
    
    precision = num_common / len(pred_tokens)
    recall = num_common / len(gt_tokens)
    
    return 2 * precision * recall / (precision + recall)


print("Metric functions defined.")

Metric functions defined.


In [37]:
# Compute all metrics
em_scores = []
f1_scores = []

for pred, gt in zip(all_predictions, ground_truths):
    em_scores.append(compute_exact_match(pred, gt))
    f1_scores.append(compute_f1(pred, gt))

avg_em = np.mean(em_scores)
avg_f1 = np.mean(f1_scores)

# Separate answerable vs unanswerable
answerable_em = []
answerable_f1 = []
unanswerable_em = []
unanswerable_f1 = []

for pred, gt, em, f1 in zip(all_predictions, ground_truths, em_scores, f1_scores):
    if gt == "":  # unanswerable
        unanswerable_em.append(em)
        unanswerable_f1.append(f1)
    else:
        answerable_em.append(em)
        answerable_f1.append(f1)

print("=" * 60)
print("HippoRAG Results on SQuAD v2")
print("=" * 60)
print(f"\nRetrieval:")
print(f"  Recall@2: {recall_at_2/len(questions):.4f}")
print(f"  Recall@5: {recall_at_5/len(questions):.4f}")
print(f"\nOverall QA:")
print(f"  Exact Match: {avg_em:.4f}")
print(f"  F1 Score:    {avg_f1:.4f}")
print(f"\nAnswerable ({len(answerable_em)} questions):")
print(f"  EM:  {np.mean(answerable_em):.4f}" if answerable_em else "  N/A")
print(f"  F1:  {np.mean(answerable_f1):.4f}" if answerable_f1 else "  N/A")
print(f"\nUnanswerable ({len(unanswerable_em)} questions):")
print(f"  EM:  {np.mean(unanswerable_em):.4f}" if unanswerable_em else "  N/A")
print(f"  F1:  {np.mean(unanswerable_f1):.4f}" if unanswerable_f1 else "  N/A")

HippoRAG Results on SQuAD v2

Retrieval:
  Recall@2: 0.6300
  Recall@5: 0.7900

Overall QA:
  Exact Match: 0.3200
  F1 Score:    0.3631

Answerable (45 questions):
  EM:  0.3778
  F1:  0.4735

Unanswerable (55 questions):
  EM:  0.2727
  F1:  0.2727


In [38]:
# qa_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cuda:0")

sim_scores = []
skipped = 0

for pred, gt in zip(all_predictions, ground_truths):
    norm_pred = normalize_answer(pred)
    norm_gt = normalize_answer(gt)
    
    # Both unanswerable
    if not norm_gt and not norm_pred:
        sim_scores.append(1.0)
        continue
    
    # Skip if ground truth is empty but prediction isn't (or vice versa)
    if not norm_gt:
        skipped += 1
        continue
    
    if not norm_pred:
        sim_scores.append(0.0)
        continue
    
    ref_emb = qa_model.encode(gt, convert_to_tensor=True)
    ans_emb = qa_model.encode(pred, convert_to_tensor=True)
    score = util.cos_sim(ref_emb, ans_emb).item()
    sim_scores.append(score)

print(f"Skipped {skipped} samples")
print(f"Valid samples: {len(sim_scores)}/{len(all_predictions)}")
print(f"Answer Similarity (cosine): {np.mean(sim_scores):.4f}")

Skipped 40 samples
Valid samples: 60/100
Answer Similarity (cosine): 0.6700
